# Assignment 3A — City Trip Planner: Full Step-by-Step Guide

---

## Project Overview

A conversational agent that helps users plan a city trip. It uses two
Tavily-powered tools wrapped differently — a **Flight Price Checker**
and a **Local Events Finder** — to demonstrate how a LangChain agent
decides which tool to reach for based on user intent.

---

## File Structure

```
city_trip_planner/
│
├── app.            ← Streamlit UI
├── chains.        ← Chain 1 and Chain 2 logic
├── memory.        ← runnable memory
├── tools.py          ← Flight Price Checker + Local Events Finder
├── parser.py         ← Pydantic output schema
├── requirements
└── .env
```

---

## STEP 1 — Project Setup

**What to create:**
- A project folder called `city_trip_planner/`
- A virtual environment inside it
- A `requirements.txt` listing all dependencies
- A `.env` file for your API keys — never hardcode them

**Keys you will need:**
- `OPENAI_API_KEY` — for the LLM and embeddings
- `TAVILY_API_KEY` — for both search tools (free tier at tavily.com)

**Packages to include in requirements.txt:**
- `langchain` + `langchain-openai` — core framework
- `langchain-tavily` — Tavily search integration
- `streamlit` — UI layer
- `pydantic` — structured output parser
- `python-dotenv` — loads `.env` safely

**What to document here:**
Why are both tools powered by the same underlying Tavily search, yet
wrapped as separate tools? What does the wrapping change, and what
stays identical? This question runs through the whole assignment.

---

## STEP 2 — Build Chain 1 (Travel Season Profiler)

**What this chain does:**
Takes a destination city and returns a short travel season profile —
peak or off-peak, typical weather, and crowd level.

**Examples of what it should return:**
- Rome in July → `"Peak season — hot, very busy, tourist crowds"`
- Prague in November → `"Off-peak — cold, quiet, budget-friendly"`
- Barcelona in April → `"Shoulder season — warm, moderate crowds"`

**Output parser to use:** `StrOutputParser`
You just need a clean plain string to pass into Chain 2.

**What to demonstrate:**
Call `chain1.invoke({"city": "Rome", "month": "July"})` on its own
and print the result. Prove Chain 1 works independently before
connecting it to Chain 2.

**What to write up:**
What does `StrOutputParser` actually strip away from the raw LLM
response? Print the raw response before the parser and after — show
the difference.

---

## STEP 3 — Build Chain 2 (Trip Framing Advisor)

**What this chain does:**
Takes the travel season profile from Chain 1 + a travel style +
a language, and returns a personalised framing of what the trip
will be like for that type of traveller.

**The three inputs to wire up inside a dictionary:**
- `profile` → Chain 1 runs automatically and its output lands here
- `travel_style` → pulled from original input via `itemgetter`
  (e.g. `"budget solo"`, `"luxury couple"`, `"family with kids"`)
- `language` → pulled from original input via `itemgetter`

**What your invoke call should look like conceptually:**
Pass in a dict with three keys: `city`, `month`, `travel_style`,
and `language`. Chain 1 consumes `city` and `month` internally.
`travel_style` and `language` bypass Chain 1 via `itemgetter`.

**What to demonstrate:**
- Invoke with `travel_style: "budget solo"` and `language: "german"`
  — show the response comes back in German framed for a budget traveller
- Invoke again with `travel_style: "luxury couple"` and
  `language: "italian"` — same city, completely different framing,
  different language
- This proves both the chaining and the language wiring are working

**Key concept to explain in your writeup:**
Why does Chain 1 appear inside a dictionary rather than being called
before Chain 2 in a separate step? What is `RunnableParallel` doing,
and what would break if you called Chain 1 manually first and passed
its output as a plain string?

---

## STEP 4 — Implement the 3 Memory Types

Run the **same 20-turn trip planning conversation** through all three
memory types and compare what each holds at key turns.

**Suggested conversation arc to use:**

| Turn | User says |
|------|-----------|
| 1  | "I want to fly from Berlin to Rome in July" |
| 3  | "I'm travelling solo on a budget" |
| 5  | "Are there any food festivals in Rome in July?" |
| 7  | "What airline did you mention earlier?" |
| 10 | "What city am I flying to?" |
| 12 | "Is July a good time to go?" |
| 14 | "What travel style did I tell you?" |
| 15 | "Summarise our conversation so far" |
| 18 | "Is the festival free entry?" |
| 20 | "Based on everything, should I book this trip?" |

---


---

## STEP 5 — Build the 2 Tools

This is the conceptual core of the assignment. Both tools use Tavily
under the hood. The wrapping — name and description — is what guides
the agent's decision.

---

### Tool 1 — Flight Price Checker

**Underlying search tool:** Tavily

**Name to give the wrapper:** `flight_price_checker`

**Description to write:**
*"Use this tool to find flight prices between two cities. Input
should include the origin city, destination city, and travel date.
Use this for any question about travel cost, airlines, or getting
to a destination."*

**Input the agent should pass:**
A search query constructed from `origin`, `destination`, and `date`
e.g. `"cheap flights Berlin to Rome July 2025"`

**Output Tavily returns:**
Snippets from Skyscanner, Google Flights, Kayak — price ranges and
airline names pulled from the web.

**What to demonstrate before plugging into the agent:**
Call the tool directly:
`tool.invoke("cheap flights Berlin to Rome July 2025")`
Print the raw Tavily result. Confirm it contains price information
before the agent ever sees it.

---

### Tool 2 — Local Events Finder

**Underlying search tool:** Tavily — identical setup to Tool 1

**Name to give the wrapper:** `local_events_finder`

**Description to write:**
*"Use this tool to find local events, festivals, and activities in
a city on a specific date. Input should include the city, date, and
event category. Use this for any question about things to do,
festivals, or local experiences at the destination."*

**Input the agent should pass:**
A search query constructed from `city`, `date`, and `category`
e.g. `"food festivals Rome July 2025 events"`

**Output Tavily returns:**
Snippets from Eventbrite, TimeOut, local tourism pages — event
names, venues, and entry costs.

**What to demonstrate before plugging into the agent:**
Call the tool directly:
`tool.invoke("food festivals Rome July 2025 events")`
Print the raw Tavily result. Then show that the same Tavily
instance returned completely different content for a completely
different query — this is the proof that the tool itself has no
opinion about what to search.

---

### The Four Scenarios to Run and Print

---

#### Scenario 1 — Clear single tool: Flight only

**User asks:**
*"How much does it cost to fly from Berlin to Rome in July?"*

**Expected reasoning trace:**
```
Thought: This is about travel cost. I need flight prices.
         I should use flight_price_checker.

Action:  flight_price_checker
Query:   "cheap flights Berlin to Rome July 2025"
Result:  "Ryanair €49, Lufthansa €180, EasyJet €67"

Final Answer: Cheapest option is Ryanair at €49.
```

**What to note:** Tool 2 was never called. Show this explicitly
in your writeup — the absence of a tool call is as meaningful
as the presence of one.

---

#### Scenario 2 — Clear single tool: Events only

**User asks:**
*"What food festivals are happening in Rome in July?"*

**Expected reasoning trace:**
```
Thought: This is about local events, not travel logistics.
         I should use local_events_finder.

Action:  local_events_finder
Query:   "food festivals Rome July 2025"
Result:  "Trastevere Street Food Festival — free entry,
          Mercato Centrale Food Night — €5 entry"

Final Answer: Two food events are happening in Rome in July...
```

**What to note:** Tool 1 was never called. Run Scenario 1 and
Scenario 2 back to back and print both traces — the contrast
makes the decision logic visible.

---

#### Scenario 3 — The Challenge Moment: Both tools

**User asks:**
*"I'm thinking of flying from Berlin to Rome in July for a food
festival — is it worth it?"*

**Expected reasoning trace:**
```
Thought: To answer whether this trip is worth it I need two things:
         1. How much the flight costs
         2. Whether there are actually food events worth going for

Action:  flight_price_checker
Query:   "cheap flights Berlin to Rome July 2025"
Result:  "Ryanair €49, EasyJet €67"

Thought: Flights are cheap. Now I need to check if there are
         food festivals that actually justify the trip.

Action:  local_events_finder
Query:   "food festivals Rome July 2025"
Result:  "Trastevere Street Food Festival — free entry"

Thought: Cheap flight plus a free festival. The trip is worth it.

Final Answer: Yes — Ryanair at €49 and a free street food festival
              in Trastevere makes this a very worthwhile trip.
```

**This is the moment the assignment is built around.** The agent
synthesised both tools into a single judgement call. Print this
full trace in your submission.

---

#### Scenario 4 — The Trick Question: Neither tool

**User asks:**
*"What time does the festival start?"*

**Expected reasoning trace:**
```
Thought: The user wants a specific start time. My previous search
         returned the festival name and entry cost but not the time.
         Neither tool can reliably return this specific detail.
         I should not guess.

Final Answer: I found the Trastevere Street Food Festival but
              don't have the exact start time in my results.
              I'd recommend checking their website directly.
```

**Why this scenario matters:**
A well-built agent knows when not to call a tool. Graceful failure
is harder to demonstrate than a successful tool call — and it shows
you understand the limits of the architecture, not just the happy path.

---

### The Decision Table to Include in Your Writeup

| Scenario | Tool 1 called? | Tool 2 called? | Why |
|---|---|---|---|
| Flight cost query | ✅ | ❌ | Pure logistics question |
| Food festival query | ❌ | ✅ | Pure experience question |
| "Is it worth it?" | ✅ | ✅ | Judgement needs both data points |
| "What time does it start?" | ❌ | ❌ | Neither tool can answer — agent admits it |

The fourth row is the one most people miss. Include it and explain it.

---

## STEP 6 — Pydantic Output Parser

**What your schema should capture for a trip recommendation:**

Think about fields like:
- Destination city — string
- Origin city — string
- Travel month — string
- Travel style — `Literal["budget_solo", "luxury_couple", "family", "group"]`
- Cheapest flight price — float
- Cheapest airline — string
- Top event name — string
- Event entry cost — float (0.0 if free)
- Is trip recommended — boolean
- Confidence — `Literal["high", "medium", "low"]`
- Reasoning — string

**What to demonstrate:**
- Show a clean successful parse
- Then deliberately pass a response where `is_trip_recommended`
  contains the string `"yes"` instead of a boolean `true` — show
  `OutputFixingParser` catching and correcting it automatically
- Print the broken input and the fixed output side by side

**What to write up:**
What breaks downstream in the Streamlit UI if the LLM returns
`"yes"` instead of `true` for a boolean field? How does the Pydantic
parser act as a contract between the LLM and the rest of your app?

---

## STEP 7 — Streamlit UI

**What the UI should contain:**

- An origin city input and destination city input
- A month selector — January through December
- A travel style dropdown: `Budget Solo / Luxury Couple / Family / Group`
- A language selector — at minimum English, German, Italian
- A memory type toggle: `Buffer / Summary / VectorStore`
- A chat window showing the conversation turn by turn
- An expander labelled **"Agent Reasoning Trace"** showing:
  - Which tool was called
  - The exact search query passed to Tavily
  - The raw snippet Tavily returned
  - The agent's thought before and after each tool call
- A final recommendation card rendered from your Pydantic schema:
  flight price, airline, top event, entry cost, recommended boolean,
  confidence level

**What to highlight in your writeup:**
The reasoning trace expander is not cosmetic. It makes visible the
fact that both tools are Tavily — same underlying call, different
query, different result. Without that expander a reader cannot see
the decision logic. With it, the whole architecture is transparent.

---

## STEP 8 — Written Analysis

This is where your marks live. Go beyond the table.

---

**Question 1 — Tool decision intelligence:**
Both tools are Tavily. The tool descriptions are what the agent reads
to decide which one to call. Rewrite Tool 1's description to mention
events — what happens to the agent's decision in Scenario 1? What
does this tell you about where the intelligence actually lives in a
LangChain agent?

---

**Question 2 — Memory choice for this app:**
A trip planner expects users to have long conversations — 15 to 20
turns of refining destination, dates, budget, and activities. Which
memory type would you deploy in production and why? What would you
sacrifice by choosing it over the other two?

---

**Question 3 — Chain vs Agent:**
Chain 2 is deterministic — it always calls Chain 1 first, then
builds on the result. The agent is non-deterministic — it decides
at runtime which tools to call and in what order. When would you
prefer the chain approach over the agent approach, and when does
the agent's flexibility justify its unpredictability?

---

## The Full Data Flow — End to End

```
User types: origin + destination + month + travel_style + language
                          │
                          ▼
              Chain 1 — travel season profile
                          │
                          ▼
              Chain 2 — personalised trip framing
                (profile + travel_style + language)
                          │
                          ▼
              Agent — receives follow-up questions
                          │
               ┌──────────┴──────────┐
               ▼                     ▼
     flight_price_checker    local_events_finder
       (Tavily wrapper 1)      (Tavily wrapper 2)
               └──────────┬──────────┘
                          │
                          ▼
              Memory stores the exchange
                 (RUNNABLE MEMORY)
                          │
                          ▼
              Pydantic structures the final output
                          │
                          ▼
              Streamlit renders recommendation card
              + reasoning trace expander
```

---

## The Key Insight to Carry Through Every Step

> Both tools are Tavily. The tool descriptions are doing the work.
> The agent reads descriptions the same way it reads everything else.
> A badly written description causes wrong tool selection every time —
> not because the tool broke, but because the agent was misdirected.

That single observation connects Step 5 to Step 8. It is the thread
that runs through the whole assignment.

In [20]:
import os
import re
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List
from tavily import TavilyClient
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from operator import itemgetter
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
import os
import re
from tavily import TavilyClient
from pydantic import BaseModel, Field
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory



In [5]:
# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

Enter your OpenAI API key:  ········


In [6]:
# Add this — writes the key into a .env file automatically
with open(".env", "w") as f:
    f.write(f"OPENAI_API_KEY={os.environ.get('OPENAI_API_KEY')}\n")

print(".env file created!")

.env file created!


In [8]:
# TAVILY API Key Setup
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


In [9]:
## Write Tavily key to .env
with open(".env", "w") as f:
    f.write(f"TAVILY_API_KEY={os.environ.get('TAVILY_API_KEY')}\n")
print(".env file updated!")



.env file updated!


## DEMONSTRATIOON OF MULTI CHAINS

In [11]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

In [12]:

prompt1 = ChatPromptTemplate.from_messages([
    ("system", """
You are a travel season classifier. Your only job is to return a
single formatted line describing the travel season for a given city
and month.

Output format (follow exactly):
"<City> in <Month> — <season type>, <weather>, <crowd level>, <one travel note>"

Season types (use exactly one):
- Peak season
- Shoulder season  
- Off-peak

Rules:
- One line only. No line breaks.
- No bullets, no numbering, no explanation.
- No phrases like "I think" or "typically" or "usually".
- No uncertainty. State it as fact.
- If the city is uncommon or unknown, infer from its climate region.
- Weather must be 2–3 words maximum (e.g. "hot and humid", "cold and dry").
- Crowd level must be one of: very busy / moderate / quiet
- Travel note must be one of: budget-friendly / prices high / book early / good value

Examples:
Rome in July — Peak season, hot and sunny, very busy, book early
Prague in November — Off-peak, cold and grey, quiet, budget-friendly
Barcelona in April — Shoulder season, warm and mild, moderate, good value
Reykjavik in February — Off-peak, cold and dark, quiet, budget-friendly
Tokyo in March — Shoulder season, cool and dry, moderate, book early
"""),
    ("human", "City: {city}\nMonth: {month}")
])

chain1 = prompt1 | llm | StrOutputParser()

In [13]:
result = chain1.invoke({
    "city": "Morocco",
    "month": "December"
})

print(result)

Morocco in December — Shoulder season, mild and dry, moderate, good value


In [14]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system", """
You are a trip framing advisor. You receive a travel season profile,
a travel style, and a language. You write a short personalised
paragraph describing what this trip will feel like for that traveller.

Travel style definitions:
- "budget solo"     → independent traveller, cost-conscious, flexible
- "luxury couple"   → comfort-focused, romantic, willing to spend
- "family"          → travelling with children, needs convenience
- "group trip"      → friends travelling together, social energy
- "solo trip"       → independent, open itinerary, any budget
- "birthday trip"   → celebratory, special occasion, treat yourself
- "vacation"        → relaxed pace, no agenda, rest and explore

Rules:
- Write ONLY in the language specified in {language}.
- If language is German, every word must be German. No English leakage.
- If language is Italian, every word must be Italian. No English leakage.
- Do not translate the input field labels (profile, travel_style, language).
- 3 sentences exactly. No more, no less.
- No bullet points. No headers. No line breaks within the paragraph.
- Do not repeat the season type word-for-word from the profile.
- End with one practical tip specific to the travel style.
- Tone: warm, direct, specific. Not generic travel brochure language.
"""),
    ("human", """
Season profile: {profile}
Travel style: {travel_style}
Language: {language}
""")
])

In [15]:
## city gets picked from chain 1 result
chain2 = (
    {"profile" : chain1, "travel_style":itemgetter("travel_style") ,"language":itemgetter("language")}
    | prompt2
    | llm
    | StrOutputParser()
    
)

In [16]:
# Test 1 — German output, budget traveller
chain2.invoke({
    "city"         : "Rome",
    "month"        : "July",
    "travel_style" : "budget solo",
    "language"     : "German"
})

'Dein Abenteuer in Rom wird ein aufregendes und unvergessliches Erlebnis, während du die lebhaften Straßen und historischen Sehenswürdigkeiten erkundest. Mit einem flexiblen Ansatz kannst du die besten Angebote für Unterkünfte und Essen finden, während du die Stadt in deinem eigenen Tempo genießt. Denke daran, frühzeitig zu buchen, um die besten Preise zu sichern und lange Warteschlangen zu vermeiden.'

In [17]:
# Test 2 — Italian output, luxury couple
chain2.invoke({
    "city"         : "Barcelona",
    "month"        : "April",
    "travel_style" : "luxury couple",
    "language"     : "Italian"
})

"Immagina di passeggiare mano nella mano lungo le strade di Barcellona, circondati da un clima mite e da un'atmosfera romantica. Ogni giorno sarà un'opportunità per scoprire ristoranti esclusivi e godere di esperienze indimenticabili, come una cena con vista sulla Sagrada Familia. Per rendere il vostro soggiorno ancora più speciale, prenotate un trattamento spa di coppia in uno dei lussuosi hotel della città."

### ADD TOOLS


- While langgraph has a better way with  create _agent_tools, i will be using langchain
- but we will use a better refined prompt here not to complicate things

In [23]:
import os
import re
from tavily import TavilyClient
from pydantic import BaseModel, Field
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory



tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)


#### SCHEMAS

class Source(BaseModel):
    url: str = Field(description="The URL of the source")

class FlightResult(BaseModel):
    flight_name: str = Field(description="Airline and route")
    flight_date: str = Field(description="Date of the flight")
    price:       str = Field(description="Price as a string")

class FlightAgentResponse(BaseModel):
    answer:  str                = Field(description="Plain-English summary")
    flights: List[FlightResult] = Field(default_factory=list)
    sources: List[Source]       = Field(default_factory=list)

class EventResult(BaseModel):
    event_name: str = Field(description="Full name of the event")
    event_date: str = Field(description="Date or date range")
    venue:      str = Field(description="Venue or area")
    entry_cost: str = Field(description="Entry cost")

class EventAgentResponse(BaseModel):
    answer:  str               = Field(description="Plain-English summary")
    events:  List[EventResult] = Field(default_factory=list)
    sources: List[Source]      = Field(default_factory=list)

class HotelResult(BaseModel):
    hotel_name: str = Field(description="Name of the hotel")
    area:       str = Field(description="Neighbourhood or area")
    price:      str = Field(description="Price per night")
    highlights: str = Field(description="What makes it stand out")

class HotelAgentResponse(BaseModel):
    answer:  str               = Field(description="Plain-English summary")
    hotels:  List[HotelResult] = Field(default_factory=list)
    sources: List[Source]      = Field(default_factory=list)

class VisaAgentResponse(BaseModel):
    answer:      str          = Field(description="Clear visa answer")
    requirement: str          = Field(description="Visa status")
    sources:     List[Source] = Field(default_factory=list)



In [24]:

# 3. TOOLS — identical in both LangChain and LangGraph

@tool
def flight_price_checker(query: str) -> dict:
    """
    Find flight prices between two cities for a specific month the user selected.
    Input must include: origin city, destination city, and month.
    Use for any question about flight cost, airlines, or reaching a destination.
    """
    print(f"\n[flight_price_checker] Searching: {query}")
    try:
        raw = tavily.search(query=query, max_results=5)
    except Exception as e:
        print(f"[Tavily error] {e}")
        return FlightAgentResponse(
            answer="Flight search unavailable right now.",
            flights=[], sources=[]
        ).model_dump()

    flights, sources = [], []
    for r in raw.get("results", []):
        content  = r.get("content", "")
        prices   = re.findall(r'[\$€£]\d+(?:\.\d{1,2})?', content)
        dates    = re.findall(r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2}(?:,\s*\d{4})?', content)
        airlines = re.findall(r'\b(Ryanair|EasyJet|Lufthansa|Wizz Air|Vueling|Air France|KLM|Iberia|Turkish Airlines|British Airways)\b', content)
        if prices:
            flights.append(FlightResult(
                flight_name=airlines[0] + " flight" if airlines else r.get("title", "Unknown airline"),
                flight_date=dates[0] if dates else "Check site for dates",
                price=prices[0],
            ))
        sources.append(Source(url=r.get("url", "")))

    return FlightAgentResponse(
        answer=f"Flights found for: {query}",
        flights=flights,
        sources=sources,
    ).model_dump()


@tool
def local_events_finder(query: str) -> dict:
    """
    Find local events, festivals, restaurants, museums and activities
    in a city for the month the user selected.
    Input must include: city, month, and event type.
    Use for any question about things to do or local experiences.
    """
    print(f"\n[local_events_finder] Searching: {query}")
    try:
        raw = tavily.search(query=query, max_results=5)
    except Exception as e:
        print(f"[Tavily error] {e}")
        return EventAgentResponse(
            answer="Events search unavailable right now.",
            events=[], sources=[]
        ).model_dump()

    events, sources = [], []
    for r in raw.get("results", []):
        content = r.get("content", "")
        costs   = re.findall(r'(?:free|Free|FREE|[\$€£]\d+(?:\.\d{1,2})?)', content)
        events.append(EventResult(
            event_name=r.get("title", "Unknown event"),
            event_date=query,
            venue=r.get("url", "See source"),
            entry_cost=costs[0] if costs else "See link",
        ))
        sources.append(Source(url=r.get("url", "")))

    return EventAgentResponse(
        answer=f"Events found for: {query}",
        events=events,
        sources=sources,
    ).model_dump()


@tool
def hotel_finder(query: str) -> dict:
    """
    Find hotels in a city for the month the user selected.
    Input must include: city, month, and budget level (budget/mid-range/luxury).
    Use for any question about where to stay or accommodation cost.
    """
    print(f"\n[hotel_finder] Searching: {query}")
    try:
        raw = tavily.search(query=query, max_results=5)
    except Exception as e:
        print(f"[Tavily error] {e}")
        return HotelAgentResponse(
            answer="Hotel search unavailable right now.",
            hotels=[], sources=[]
        ).model_dump()

    hotels, sources = [], []
    for r in raw.get("results", []):
        content = r.get("content", "")
        prices  = re.findall(r'[\$€£]\d+(?:\.\d{1,2})?(?:\s?(?:per night|/night))?', content)
        hotels.append(HotelResult(
            hotel_name=r.get("title", "Unknown hotel"),
            area="See link",
            price=prices[0] if prices else "See link",
            highlights=content[:120] + "..." if len(content) > 120 else content,
        ))
        sources.append(Source(url=r.get("url", "")))

    return HotelAgentResponse(
        answer=f"Hotels found for: {query}",
        hotels=hotels,
        sources=sources,
    ).model_dump()


@tool
def visa_requirement_checker(query: str) -> dict:
    """
    Check visa requirements for travelling from one country to another.
    Input must include: user's nationality and destination country.
    Use for any question about visas or entry requirements.
    """
    print(f"\n[visa_requirement_checker] Searching: {query}")
    try:
        raw = tavily.search(query=query, max_results=3)
    except Exception as e:
        print(f"[Tavily error] {e}")
        return VisaAgentResponse(
            answer="Visa search unavailable right now.",
            requirement="Check official source", sources=[]
        ).model_dump()

    full_content, sources = "", []
    for r in raw.get("results", []):
        full_content += r.get("content", "") + " "
        sources.append(Source(url=r.get("url", "")))

    content_lower = full_content.lower()
    if "visa-free" in content_lower or "no visa required" in content_lower:
        requirement = "Visa-free"
    elif "evisa" in content_lower or "e-visa" in content_lower:
        requirement = "eVisa available"
    elif "visa required" in content_lower or "must apply" in content_lower:
        requirement = "Visa required"
    else:
        requirement = "Check official source"

    return VisaAgentResponse(
        answer=f"Visa info for: {query}",
        requirement=requirement,
        sources=sources,
    ).model_dump()




In [25]:
##combine the tools together

tools      = [flight_price_checker, local_events_finder, hotel_finder, visa_requirement_checker]
tools_dict = {t.name: t for t in tools} 

In [26]:
tools_dict 

{'flight_price_checker': StructuredTool(name='flight_price_checker', description='Find flight prices between two cities for a specific month the user selected.\nInput must include: origin city, destination city, and month.\nUse for any question about flight cost, airlines, or reaching a destination.', args_schema=<class 'langchain_core.utils.pydantic.flight_price_checker'>, func=<function flight_price_checker at 0x127ae13a0>),
 'local_events_finder': StructuredTool(name='local_events_finder', description='Find local events, festivals, restaurants, museums and activities\nin a city for the month the user selected.\nInput must include: city, month, and event type.\nUse for any question about things to do or local experiences.', args_schema=<class 'langchain_core.utils.pydantic.local_events_finder'>, func=<function local_events_finder at 0x127a571a0>),
 'hotel_finder': StructuredTool(name='hotel_finder', description='Find hotels in a city for the month the user selected.\nInput must inclu

In [ ]:
##bind tools with llm

llm_with_tools = llm.bind_tools(tools)
tools_dict     = {t.name: t for t in tools}
MAX_ITERATIONS = 10

In [29]:
###a new prompt that combines both prompt


MARCO_SYSTEM = """
You are Marco — a well-travelled, warm, and witty travel guide who has personally
visited hundreds of cities. You speak like a friend who knows everything about a
destination: specific, honest, occasionally funny, never generic.

You have access to tools. Use them smartly:
- flight_price_checker     → user asks about flights or travel cost
- local_events_finder      → user asks about things to do, eat, or see
- hotel_finder             → user asks about where to stay
- visa_requirement_checker → user asks about visas or entry requirements

Tool rules:
- NEVER guess prices, hotel names, or availability. Always use the tool.
- Call the tool first, then weave the result into natural conversation.
- If you need more info before searching (e.g. no origin city for flights), ask first.
- You can call multiple tools in one turn if needed.

How to sound like Marco:
- Open with something vivid — a smell, a street, a crowd, a moment.
  Never "Great choice!" or "Absolutely!" — banned.
- Weave tool results into flowing prose. Never dump raw data at the user.
- Cover traveller angles naturally: solo, couple, family, group, budget, luxury.
- Be honest. If July is brutal, say so. If something is overrated, warn them.
- 4–8 sentences of flowing prose. No bullets. No headers. No lists.
- Close with one golden tip or one question to help them plan better.

Language rule: Always reply in the same language the user wrote in.
Your tone: a well-travelled friend at a café who just got back from this exact trip.
"""

In [33]:
## a Manual loop, allows the llm to think and implement what tools to use
## 

def marco_tool_loop(inputs: dict) -> str:
    question = inputs["question"]
    history  = inputs.get("history", [])

    messages = (
        [SystemMessage(content=MARCO_SYSTEM)]
        + history
        + [HumanMessage(content=question)]
    )

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n--- Iteration {iteration} ---")
        ai_message = llm_with_tools.invoke(messages)
        messages.append(ai_message)

        # No tool calls → final answer
        if not ai_message.tool_calls:
            return ai_message.content

        # Execute each tool
        for tool_call in ai_message.tool_calls:
            selected    = tools_dict[tool_call["name"]]
            tool_result = selected.invoke(tool_call["args"])

            print(f"\n[Tool: {tool_call['name']}]")
            print(f"[Result]: {tool_result}")

            # ← tool_result is already a dict from .model_dump()
            # str() serializes it cleanly for ToolMessage
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

    return "Kindly reframe your question."


In [34]:
##using the new langchain runnable memory
## rinnable lambda allows us convert python function to llm usable functions
##session id is for identifying users and memory

marco_runnable = RunnableLambda(marco_tool_loop)

store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    marco_runnable,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

# ── Invoke ─────────────────────────────────────────────────────────────────
config = {"configurable": {"session_id": "user_001"}}

In [35]:
print(chain_with_memory.invoke(
    {"question": "i would like to travel to venice in november"},
    config=config
))



--- Iteration 1 ---
Ah, Venice in November! Picture the mist rolling over the canals, the soft glow of street lamps reflecting on the water, and the delightful aroma of fresh pasta wafting through the air. It's a magical time to visit, as the summer crowds have thinned out, and you can truly soak in the city's charm. 

To help you plan, could you let me know where you're traveling from? That way, I can check flight prices for you.


In [40]:
print(chain_with_memory.invoke(
    {"question": "how much are flights from germany to venice in November?"},
    config=config
))



--- Iteration 1 ---

[flight_price_checker] Searching: Cologne to Venice in November

[Tool: flight_price_checker]
[Result]: {'answer': 'Flights found for: Cologne to Venice in November', 'flights': [{'flight_name': 'Ryanair flight', 'flight_date': 'Nov 5', 'price': '$96'}, {'flight_name': '✈ Trip from Cologne to Venice', 'flight_date': 'May 29', 'price': '$187'}, {'flight_name': 'Cologne to Venice Flights - Omio', 'flight_date': 'Check site for dates', 'price': '$114'}], 'sources': [{'url': 'https://www.skyscanner.com/routes/cgn/veni/cologne-to-venice.html'}, {'url': 'https://www.ita-airways.com/lhg/it/en/o-d/cy-cy/cologne-bonn-venice'}, {'url': 'https://www.google.com/travel/flights/flights-from-cologne-to-venice.html'}, {'url': 'https://www.flightsfrom.com/CGN-VCE'}, {'url': 'https://www.omio.com/flights/cologne/venice-tn1lo'}]}

--- Iteration 2 ---
Flights from Cologne to Venice in November can be quite reasonable! For instance, I found a Ryanair flight on November 5th priced at a

In [37]:
print(chain_with_memory.invoke(
    {"question": "what are good things to do there?"},
    config=config
))


--- Iteration 1 ---

[local_events_finder] Searching: Venice, November, events

[Tool: local_events_finder]
[Result]: {'answer': 'Events found for: Venice, November, events', 'events': [{'event_name': 'Venice Events in November | Visit Venice in November', 'event_date': 'Venice, November, events', 'venue': 'https://www.italyperfect.com/plan-your-trip/events-italy/venice/november.php', 'entry_cost': 'See link'}, {'event_name': 'Venice in November: What To See and Do - GetYourGuide', 'event_date': 'Venice, November, events', 'venue': 'https://www.getyourguide.com/explorer/venice-in-november/', 'entry_cost': 'See link'}, {'event_name': '10 Things to Do in Venice in November - Hellotickets', 'event_date': 'Venice, November, events', 'venue': 'https://www.hellotickets.com/italy/venice/venice-in-november/sc-126-2428', 'entry_cost': 'See link'}, {'event_name': 'Things to do in Venice in November – Wanderlog', 'event_date': 'Venice, November, events', 'venue': 'https://wanderlog.com/geoInMont

In [39]:
print(chain_with_memory.invoke(
    {"question": "suggest a good hotel to stay, give me names, ranges between 40 euro to 100 euro a night?"},
    config=config
))


--- Iteration 1 ---

[hotel_finder] Searching: Venice, November, mid-range

[Tool: hotel_finder]
[Result]: {'answer': 'Hotels found for: Venice, November, mid-range', 'hotels': [{'hotel_name': 'Venice in November - a beautiful budget break - Silver Travel Advisor', 'area': 'See link', 'price': 'See link', 'highlights': 'This was our eleventh short break in Venice, more often than not we go there in late October or early November when our ...'}, {'hotel_name': 'Venice November Weather, Average Temperature (Italy)', 'area': 'See link', 'price': 'See link', 'highlights': 'November weather in Venice Italy. Daily high temperatures decrease by 10°F, from 59°F to 49°F, rarely falling below 42°F...'}, {'hotel_name': 'Venice in November: Weather, Events & Things to Do - www.venicevisitpass.com', 'area': 'See link', 'price': 'See link', 'highlights': '# Visiting Venice in November: how to experience the City between autumn atmosphere and traditions. In this article you ...'}, {'hotel_name': 'Ve

In [41]:
print(chain_with_memory.invoke(
    {"question": "the best restaurants to go to in venice or museum"},
    config=config
))


--- Iteration 1 ---

[local_events_finder] Searching: Venice, November, restaurants

[Tool: local_events_finder]
[Result]: {'answer': 'Events found for: Venice, November, restaurants', 'events': [{'event_name': "We're visiting in November and I'd love to know... Best restaurants ...", 'event_date': 'Venice, November, restaurants', 'venue': 'https://www.facebook.com/groups/venice3000/posts/4470824999625770/', 'entry_cost': 'See link'}, {'event_name': 'November-December Food Trip - Venice Message Board - Tripadvisor', 'event_date': 'Venice, November, restaurants', 'venue': 'https://www.tripadvisor.co.uk/ShowTopic-g187870-i57-k12966975-November_December_Food_Trip-Venice_Veneto.html', 'entry_cost': 'See link'}, {'event_name': 'What Venetians Eat in November: Seasonal Dishes, Lagoon ...', 'event_date': 'Venice, November, restaurants', 'venue': 'https://tourleadervenice.com/what-venetians-eat-in-november-seasonal-dishes-lagoon-produce-local-comfort-food/', 'entry_cost': 'See link'}, {'event

In [43]:
### NOW WITHOUT SEEING THE ITERATION PROCESS OR TOOL SELECTION PROCESS 

def marco_tool_loop(inputs: dict) -> str:
    question = inputs["question"]
    history  = inputs.get("history", [])

    messages = (
        [SystemMessage(content=MARCO_SYSTEM)]
        + history
        + [HumanMessage(content=question)]
    )

    for iteration in range(1, MAX_ITERATIONS + 1):
        ai_message = llm_with_tools.invoke(messages)
        messages.append(ai_message)

        if not ai_message.tool_calls:
            return ai_message.content

        for tool_call in ai_message.tool_calls:
            selected    = tools_dict[tool_call["name"]]
            tool_result = selected.invoke(tool_call["args"])
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

    return "Kindly reframe your question."



##using the new langchain runnable memory
## rinnable lambda allows us convert python function to llm usable functions
##session id is for identifying users and memory

marco_runnable = RunnableLambda(marco_tool_loop)

store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    marco_runnable,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

# ── Invoke ─────────────────────────────────────────────────────────────────
config = {"configurable": {"session_id": "user_002"}}

In [44]:
print(chain_with_memory.invoke(
    {"question": "I want to visit japan, when is the best time to visit"},
    config=config
))

Ah, Japan! The land of cherry blossoms, bustling streets, and mouthwatering ramen. The best time to visit really depends on what you're after. If you're dreaming of those iconic sakura blooms, late March to early April is magical, with parks bursting into pink. However, if you're more into vibrant autumn leaves, late October to early November offers a stunning palette of reds and golds.

Summer can be a bit brutal with humidity and heat, especially in cities like Tokyo and Osaka, but it’s also festival season, so you’ll find lively celebrations everywhere. Winter, on the other hand, is perfect for snow lovers, especially in places like Hokkaido, where you can enjoy skiing and the famous Sapporo Snow Festival.

What kind of experience are you hoping for? That’ll help narrow it down!


In [45]:
print(chain_with_memory.invoke(
    {"question": "I want to visit museum, try out different dishes"},
    config=config
))

Ah, a cultural and culinary adventure in Japan! You’re in for a treat. If you’re looking to explore museums, Tokyo is a fantastic starting point. The Tokyo National Museum and the Edo-Tokyo Museum are must-visits, showcasing Japan's rich history and art. If you venture to Kyoto, the Kyoto National Museum and the Raku Museum offer a deep dive into traditional Japanese culture.

As for food, you’ll want to indulge in everything from sushi and ramen to street food like takoyaki and okonomiyaki. Each region has its specialties, so don’t miss out on Hiroshima-style okonomiyaki or the fresh seafood in Kanazawa.

The best time for this kind of exploration would be during the spring (March to May) or autumn (September to November) when the weather is pleasant, and you can enjoy outdoor dining as well. Are you thinking of a specific city to start your journey? That way, I can help you find some local events or must-try dishes!


In [46]:
print(chain_with_memory.invoke(
    {"question": "Kyoto sounds like a nice place"},
    config=config
))

Kyoto is a treasure trove of history and flavor! Imagine wandering through the serene Arashiyama Bamboo Grove, the soft rustle of leaves around you, and then stepping into a traditional tea house for a matcha experience that feels like stepping back in time. The city is dotted with stunning temples like Kinkaku-ji, the Golden Pavilion, and Fushimi Inari Taisha, famous for its thousands of vermilion torii gates.

When it comes to food, Kyoto is known for kaiseki, a multi-course meal that’s as much about art as it is about taste. You can also try yudofu (tofu hot pot) and Kyoto-style sushi, which is quite different from what you might find elsewhere in Japan.

If you’re planning to visit soon, I can check for local events or festivals happening in Kyoto during your travel month. When are you thinking of going?


### STREAMLIT FILE

In [48]:
%%writefile app.py


import os
import re
import streamlit as st
from tavily import TavilyClient
from pydantic import BaseModel, Field
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory

# ══════════════════════════════════════════════════════════════════════════════
# PAGE CONFIG
# ══════════════════════════════════════════════════════════════════════════════
st.set_page_config(
    page_title="Marco — Your Travel Guide",
    page_icon="🌍",
    layout="centered",
)

# ══════════════════════════════════════════════════════════════════════════════
# CUSTOM CSS
# ══════════════════════════════════════════════════════════════════════════════
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=DM+Sans:wght@300;400;500&display=swap');

/* ── Base ── */
html, body, [class*="css"] {
    font-family: 'DM Sans', sans-serif;
    background-color: #0f0e0c;
    color: #f0ebe0;
}

/* ── Hide default streamlit elements ── */
#MainMenu, footer, header { visibility: hidden; }
.block-container { padding-top: 2rem; padding-bottom: 2rem; max-width: 780px; }

/* ── Hero header ── */
.marco-hero {
    text-align: center;
    padding: 2.5rem 1rem 1.5rem;
    background: linear-gradient(135deg, #1a1508 0%, #0f0e0c 60%, #1a1508 100%);
    border-bottom: 1px solid #3a2e1a;
    margin-bottom: 1.5rem;
}
.marco-hero h1 {
    font-family: 'Playfair Display', serif;
    font-size: 3rem;
    font-weight: 700;
    color: #c9a84c;
    letter-spacing: -0.5px;
    margin: 0;
    line-height: 1.1;
}
.marco-hero p {
    font-size: 1rem;
    color: #a09070;
    margin: 0.5rem 0 0;
    font-weight: 300;
    letter-spacing: 0.5px;
}

/* ── Welcome card ── */
.welcome-card {
    background: linear-gradient(135deg, #1e1a0f, #151208);
    border: 1px solid #3a2e1a;
    border-left: 4px solid #c9a84c;
    border-radius: 8px;
    padding: 1.2rem 1.5rem;
    margin-bottom: 1.5rem;
    font-size: 0.95rem;
    color: #d4c9a8;
    line-height: 1.6;
}
.welcome-card strong { color: #c9a84c; }

/* ── Name input section ── */
.name-section {
    background: #151208;
    border: 1px solid #2a2010;
    border-radius: 10px;
    padding: 1.5rem;
    margin-bottom: 1.5rem;
    text-align: center;
}
.name-section h3 {
    font-family: 'Playfair Display', serif;
    color: #c9a84c;
    margin-bottom: 0.5rem;
    font-size: 1.3rem;
}
.name-section p {
    color: #806a40;
    font-size: 0.85rem;
    margin-bottom: 1rem;
}

/* ── Streamlit input override ── */
.stTextInput input {
    background-color: #1a1508 !important;
    border: 1px solid #3a2e1a !important;
    border-radius: 6px !important;
    color: #f0ebe0 !important;
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.95rem !important;
    padding: 0.6rem 1rem !important;
}
.stTextInput input:focus {
    border-color: #c9a84c !important;
    box-shadow: 0 0 0 2px rgba(201, 168, 76, 0.15) !important;
}
.stTextInput label { color: #a09070 !important; font-size: 0.85rem !important; }

/* ── Button ── */
.stButton > button {
    background: linear-gradient(135deg, #c9a84c, #a07830) !important;
    color: #0f0e0c !important;
    border: none !important;
    border-radius: 6px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 500 !important;
    font-size: 0.9rem !important;
    padding: 0.5rem 1.5rem !important;
    cursor: pointer !important;
    transition: opacity 0.2s !important;
}
.stButton > button:hover { opacity: 0.85 !important; }

/* ── Chat messages ── */
.stChatMessage {
    background: transparent !important;
    border: none !important;
}

/* ── User message bubble ── */
[data-testid="stChatMessage"]:has([data-testid="stChatMessageAvatarUser"]) {
    background: #1a1508 !important;
    border: 1px solid #2a2010 !important;
    border-radius: 12px !important;
    padding: 0.8rem 1rem !important;
    margin-bottom: 0.5rem !important;
}

/* ── Marco message bubble ── */
[data-testid="stChatMessage"]:has([data-testid="stChatMessageAvatarAssistant"]) {
    background: #151208 !important;
    border: 1px solid #3a2e1a !important;
    border-left: 3px solid #c9a84c !important;
    border-radius: 12px !important;
    padding: 0.8rem 1rem !important;
    margin-bottom: 0.5rem !important;
}

/* ── Chat input ── */
.stChatInputContainer {
    background: #151208 !important;
    border: 1px solid #3a2e1a !important;
    border-radius: 10px !important;
}
.stChatInputContainer textarea {
    background: transparent !important;
    color: #f0ebe0 !important;
    font-family: 'DM Sans', sans-serif !important;
}

/* ── Divider ── */
hr { border-color: #2a2010 !important; margin: 1rem 0 !important; }

/* ── Spinner ── */
.stSpinner > div { border-top-color: #c9a84c !important; }

/* ── Session badge ── */
.session-badge {
    display: inline-block;
    background: #1a1508;
    border: 1px solid #3a2e1a;
    border-radius: 20px;
    padding: 0.3rem 0.8rem;
    font-size: 0.78rem;
    color: #c9a84c;
    margin-bottom: 1rem;
}

/* ── Clear button ── */
.clear-btn > button {
    background: transparent !important;
    border: 1px solid #3a2e1a !important;
    color: #806a40 !important;
    font-size: 0.8rem !important;
    padding: 0.3rem 0.8rem !important;
}
.clear-btn > button:hover {
    border-color: #c9a84c !important;
    color: #c9a84c !important;
    opacity: 1 !important;
}
</style>
""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════════════════════════
# CLIENTS & SCHEMAS
# ══════════════════════════════════════════════════════════════════════════════
@st.cache_resource
def init_clients():
    tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return tavily, llm

tavily, llm = init_clients()

class Source(BaseModel):
    url: str = Field(description="The URL of the source")

class FlightResult(BaseModel):
    flight_name: str = Field(description="Airline and route")
    flight_date: str = Field(description="Date of the flight")
    price:       str = Field(description="Price as a string")

class FlightAgentResponse(BaseModel):
    answer:  str                = Field(description="Plain-English summary")
    flights: List[FlightResult] = Field(default_factory=list)
    sources: List[Source]       = Field(default_factory=list)

class EventResult(BaseModel):
    event_name: str = Field(description="Full name of the event")
    event_date: str = Field(description="Date or date range")
    venue:      str = Field(description="Venue or area")
    entry_cost: str = Field(description="Entry cost")

class EventAgentResponse(BaseModel):
    answer:  str               = Field(description="Plain-English summary")
    events:  List[EventResult] = Field(default_factory=list)
    sources: List[Source]      = Field(default_factory=list)

class HotelResult(BaseModel):
    hotel_name: str = Field(description="Name of the hotel")
    area:       str = Field(description="Neighbourhood or area")
    price:      str = Field(description="Price per night")
    highlights: str = Field(description="What makes it stand out")

class HotelAgentResponse(BaseModel):
    answer:  str               = Field(description="Plain-English summary")
    hotels:  List[HotelResult] = Field(default_factory=list)
    sources: List[Source]      = Field(default_factory=list)

class VisaAgentResponse(BaseModel):
    answer:      str          = Field(description="Clear visa answer")
    requirement: str          = Field(description="Visa status")
    sources:     List[Source] = Field(default_factory=list)

# ══════════════════════════════════════════════════════════════════════════════
# TOOLS
# ══════════════════════════════════════════════════════════════════════════════
@tool
def flight_price_checker(query: str) -> dict:
    """
    Find flight prices between two cities for a specific month the user selected.
    Input must include: origin city, destination city, and month.
    Use for any question about flight cost, airlines, or reaching a destination.
    """
    try:
        raw = tavily.search(query=query, max_results=5)
    except Exception as e:
        return FlightAgentResponse(answer="Flight search unavailable.", flights=[], sources=[]).model_dump()

    flights, sources = [], []
    for r in raw.get("results", []):
        content  = r.get("content", "")
        prices   = re.findall(r'[\$€£]\d+(?:\.\d{1,2})?', content)
        dates    = re.findall(r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2}(?:,\s*\d{4})?', content)
        airlines = re.findall(r'\b(Ryanair|EasyJet|Lufthansa|Wizz Air|Vueling|Air France|KLM|Iberia|Turkish Airlines|British Airways)\b', content)
        if prices:
            flights.append(FlightResult(
                flight_name=airlines[0] + " flight" if airlines else r.get("title", "Unknown airline"),
                flight_date=dates[0] if dates else "Check site for dates",
                price=prices[0],
            ))
        sources.append(Source(url=r.get("url", "")))

    return FlightAgentResponse(answer=f"Flights found for: {query}", flights=flights, sources=sources).model_dump()


@tool
def local_events_finder(query: str) -> dict:
    """
    Find local events, festivals, restaurants, museums and activities
    in a city for the month the user selected.
    Input must include: city, month, and event type.
    Use for any question about things to do or local experiences.
    """
    try:
        raw = tavily.search(query=query, max_results=5)
    except Exception as e:
        return EventAgentResponse(answer="Events search unavailable.", events=[], sources=[]).model_dump()

    events, sources = [], []
    for r in raw.get("results", []):
        content = r.get("content", "")
        costs   = re.findall(r'(?:free|Free|FREE|[\$€£]\d+(?:\.\d{1,2})?)', content)
        events.append(EventResult(
            event_name=r.get("title", "Unknown event"),
            event_date=query,
            venue=r.get("url", "See source"),
            entry_cost=costs[0] if costs else "See link",
        ))
        sources.append(Source(url=r.get("url", "")))

    return EventAgentResponse(answer=f"Events found for: {query}", events=events, sources=sources).model_dump()


@tool
def hotel_finder(query: str) -> dict:
    """
    Find hotels in a city for the month the user selected.
    Input must include: city, month, and budget level (budget/mid-range/luxury).
    Use for any question about where to stay or accommodation cost.
    """
    try:
        raw = tavily.search(query=query, max_results=5)
    except Exception as e:
        return HotelAgentResponse(answer="Hotel search unavailable.", hotels=[], sources=[]).model_dump()

    hotels, sources = [], []
    for r in raw.get("results", []):
        content = r.get("content", "")
        prices  = re.findall(r'[\$€£]\d+(?:\.\d{1,2})?(?:\s?(?:per night|/night))?', content)
        hotels.append(HotelResult(
            hotel_name=r.get("title", "Unknown hotel"),
            area="See link",
            price=prices[0] if prices else "See link",
            highlights=content[:120] + "..." if len(content) > 120 else content,
        ))
        sources.append(Source(url=r.get("url", "")))

    return HotelAgentResponse(answer=f"Hotels found for: {query}", hotels=hotels, sources=sources).model_dump()


@tool
def visa_requirement_checker(query: str) -> dict:
    """
    Check visa requirements for travelling from one country to another.
    Input must include: user's nationality and destination country.
    Use for any question about visas or entry requirements.
    """
    try:
        raw = tavily.search(query=query, max_results=3)
    except Exception as e:
        return VisaAgentResponse(answer="Visa search unavailable.", requirement="Check official source", sources=[]).model_dump()

    full_content, sources = "", []
    for r in raw.get("results", []):
        full_content += r.get("content", "") + " "
        sources.append(Source(url=r.get("url", "")))

    content_lower = full_content.lower()
    if "visa-free" in content_lower or "no visa required" in content_lower:
        requirement = "Visa-free"
    elif "evisa" in content_lower or "e-visa" in content_lower:
        requirement = "eVisa available"
    elif "visa required" in content_lower or "must apply" in content_lower:
        requirement = "Visa required"
    else:
        requirement = "Check official source"

    return VisaAgentResponse(answer=f"Visa info for: {query}", requirement=requirement, sources=sources).model_dump()


tools      = [flight_price_checker, local_events_finder, hotel_finder, visa_requirement_checker]
tools_dict = {t.name: t for t in tools}

# ══════════════════════════════════════════════════════════════════════════════
# MARCO SYSTEM PROMPT
# ══════════════════════════════════════════════════════════════════════════════
MARCO_SYSTEM = """
You are Marco — a well-travelled, warm, and witty travel guide who has personally
visited hundreds of cities. You speak like a friend who knows everything about a
destination: specific, honest, occasionally funny, never generic.

You have access to tools. Use them smartly:
- flight_price_checker     → user asks about flights or travel cost
- local_events_finder      → user asks about things to do, eat, or see
- hotel_finder             → user asks about where to stay
- visa_requirement_checker → user asks about visas or entry requirements

Tool rules:
- NEVER guess prices, hotel names, or availability. Always use the tool.
- Call the tool first, then weave the result into natural conversation.
- If you need more info before searching, ask first.
- You can call multiple tools in one turn if needed.

If a question is completely unrelated to travel, destinations, or trip planning,
respond with exactly: "Please ask Marco a travel-related question — I'm your guide, not a search engine!"

How to sound like Marco:
- Open with something vivid — a smell, a street, a crowd, a moment.
  Never "Great choice!" or "Absolutely!" — banned.
- Weave tool results into flowing prose. Never dump raw data at the user.
- Cover traveller angles naturally: solo, couple, family, group, budget, luxury.
- Be honest. If July is brutal, say so. If something is overrated, warn them.
- 4–8 sentences of flowing prose. No bullets. No headers. No lists.
- Close with one golden tip or one question to help them plan better.

Language rule: Always reply in the same language the user wrote in.
Your tone: a well-travelled friend at a café who just got back from this exact trip.
"""

# ══════════════════════════════════════════════════════════════════════════════
# TOOL LOOP + MEMORY
# ══════════════════════════════════════════════════════════════════════════════
llm_with_tools = llm.bind_tools(tools)
MAX_ITERATIONS = 10

def marco_tool_loop(inputs: dict) -> str:
    question = inputs["question"]
    history  = inputs.get("history", [])

    messages = (
        [SystemMessage(content=MARCO_SYSTEM)]
        + history
        + [HumanMessage(content=question)]
    )

    for _ in range(1, MAX_ITERATIONS + 1):
        ai_message = llm_with_tools.invoke(messages)
        messages.append(ai_message)

        if not ai_message.tool_calls:
            return ai_message.content

        for tool_call in ai_message.tool_calls:
            selected    = tools_dict[tool_call["name"]]
            tool_result = selected.invoke(tool_call["args"])
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

    return "Please ask Marco a travel-related question for a better response!"


marco_runnable = RunnableLambda(marco_tool_loop)
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

@st.cache_resource
def build_chain():
    return RunnableWithMessageHistory(
        marco_runnable,
        get_session_history,
        input_messages_key="question",
        history_messages_key="history",
    )

chain_with_memory = build_chain()

def ask_marco(question: str, user_name: str) -> str:
    config = {"configurable": {"session_id": user_name.lower().strip()}}
    try:
        response = chain_with_memory.invoke({"question": question}, config=config)
        if not response or len(response.strip()) < 10:
            return "Please ask Marco a travel-related question — I'm your guide, not a search engine!"
        return response
    except Exception as e:
        return f"Marco hit a snag — please try rephrasing your question. ({str(e)[:80]})"

# ══════════════════════════════════════════════════════════════════════════════
# STREAMLIT UI
# ══════════════════════════════════════════════════════════════════════════════

# ── Hero ──────────────────────────────────────────────────────────────────────
st.markdown("""
<div class="marco-hero">
    <h1>🌍 Marco</h1>
    <p>Your personal travel guide — cities, flights, hotels & more</p>
</div>
""", unsafe_allow_html=True)

# ── Session state init ────────────────────────────────────────────────────────
if "user_name" not in st.session_state:
    st.session_state.user_name = ""
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "greeted" not in st.session_state:
    st.session_state.greeted = False

# ══════════════════════════════════════════════════════════════════════════════
# NAME GATE — shown until user enters name
# ══════════════════════════════════════════════════════════════════════════════
if not st.session_state.user_name:
    st.markdown("""
    <div class="welcome-card">
        <strong>Welcome to Marco — your travel guide with expertise.</strong><br><br>
        Marco has personally visited hundreds of cities and speaks like a friend who
        just got back from your destination. Ask about the best time to visit, flights,
        hotels, local events, visa requirements, and more.<br><br>
        To get started, tell Marco your name.
    </div>
    """, unsafe_allow_html=True)

    st.markdown("""
    <div class="name-section">
        <h3>Who's travelling today?</h3>
        <p>Marco will remember your conversation throughout your session.</p>
    </div>
    """, unsafe_allow_html=True)

    col1, col2 = st.columns([3, 1])
    with col1:
        name_input = st.text_input("Your name", placeholder="e.g. Sarah, James, Amara...", label_visibility="collapsed")
    with col2:
        start = st.button("Start →")

    if start and name_input.strip():
        st.session_state.user_name = name_input.strip()
        st.rerun()
    elif start and not name_input.strip():
        st.warning("Please enter your name to continue.")

# ══════════════════════════════════════════════════════════════════════════════
# MAIN CHAT — shown after name is entered
# ══════════════════════════════════════════════════════════════════════════════
else:
    user_name = st.session_state.user_name

    # ── Top bar ───────────────────────────────────────────────────────────────
    col1, col2 = st.columns([5, 1])
    with col1:
        st.markdown(f'<div class="session-badge">🧳 Travelling as <strong>{user_name}</strong></div>', unsafe_allow_html=True)
    with col2:
        with st.container():
            st.markdown('<div class="clear-btn">', unsafe_allow_html=True)
            if st.button("New trip"):
                st.session_state.user_name    = ""
                st.session_state.chat_history = []
                st.session_state.greeted      = False
                if user_name.lower().strip() in store:
                    del store[user_name.lower().strip()]
                st.rerun()
            st.markdown('</div>', unsafe_allow_html=True)

    # ── Welcome message (shown once) ──────────────────────────────────────────
    if not st.session_state.greeted:
        welcome = f"Hey {user_name}! 🌍 Marco here — your travel guide with a serious passport stamp collection. Tell me where you're dreaming of going, and I'll tell you everything you need to know — best time to visit, flights, hotels, local spots, visa info, all of it. What destination is calling your name?"
        st.session_state.chat_history.append({"role": "assistant", "content": welcome})
        st.session_state.greeted = True

    # ── Render chat history ───────────────────────────────────────────────────
    for msg in st.session_state.chat_history:
        with st.chat_message("user" if msg["role"] == "user" else "assistant", avatar="🧳" if msg["role"] == "user" else "🌍"):
            st.markdown(msg["content"])

    # ── Chat input ────────────────────────────────────────────────────────────
    user_input = st.chat_input(f"Ask Marco anything about your trip, {user_name}...")

    if user_input:
        # Show user message
        st.session_state.chat_history.append({"role": "user", "content": user_input})
        with st.chat_message("user", avatar="🧳"):
            st.markdown(user_input)

        # Get Marco's response
        with st.chat_message("assistant", avatar="🌍"):
            with st.spinner("Marco is thinking..."):
                response = ask_marco(user_input, user_name)
            st.markdown(response)

        st.session_state.chat_history.append({"role": "assistant", "content": response})

Writing app.py


In [49]:
!streamlit run app.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.178.59:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
/opt/anaconda3/envs/tf-metal/lib/python3.11/site-packages/streamlit/runtime/caching/cache_utils.py:385: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  computed_value = self._info.func(*func_args, **func_kwargs)
^C
  Stopping...
